In [3]:
import numpy as np
import pandas as pd


def celsius(f):
    return (f - 32) * 5 / 9


def m_2(pes):
    return pes * 0.0929


def outliers_kwh(df: pd.DataFrame):
    Q1 = df["kwh"].quantile(0.25)
    Q3 = df["kwh"].quantile(0.75)
    IQR = Q3 - Q1

    limite_cima = Q3 + 1.5 * IQR
    limite_baixo = Q1 - 1.5 * IQR

    return df[(df["kwh"] >= limite_baixo) & (df["kwh"] <= limite_cima)]


csv_path: str = "dados/recs2020_public_v7.csv"
url_rec = "https://www.eia.gov/consumption/residential/data/2020/csv/recs2020_public_v7.csv"

colunas = {
    # Consumo:
    "KWH": {"nome_coluna": "kwh", "info": "kWh (quilowatt-hora)"},
    "NHSLDMEM": {"nome_coluna": "moradores", "info": "Número de moradores"},
    # Temperatura:
    "HDD65": {
        "nome_coluna": "graus_dia_frio_f",
        "info": "Graus-dia de frio (65 °F) em 2020",
    },
    "CDD65": {
        "nome_coluna": "graus_dia_calor_f",
        "info": "Graus-dia de calor (65 °F) em 2020",
    },
    "TEMPHOME": {
        "nome_coluna": "temp_inverno_dia_c",
        "info": "Temperatura no inverno, de dia, com gente em casa (°C)",
    },
    "TEMPNITE": {
        "nome_coluna": "temp_inverno_noite_c",
        "info": "Temperatura no inverno, à noite (°C)",
    },
    # A Casa:
    "TOTROOMS": {
        "nome_coluna": "total_comodos",
        "info": "Total de cômodos, sem contar banheiros",
    },
    "SQFTEST": {"nome_coluna": "area_casa_m2", "info": "Área da casa (m²)"},
    # Número de Aparelhos:
    "NUMCFAN": {"nome_coluna": "ventiladores_teto", "info": "Ventiladores de teto"},
    "TVCOLOR": {"nome_coluna": "tvs", "info": "TVs"},
    "NUMFRIG": {"nome_coluna": "geladeiras", "info": "Geladeiras"},
    "NUMFREEZ": {"nome_coluna": "freezers", "info": "Freezers separados"},
    "MICRO": {"nome_coluna": "microondas", "info": "Micro-ondas"},
    # Tem Aparelhos (1 sim, 0 não):
    "DISHWASH": {"nome_coluna": "lava_louca", "info": "Lava-louça"},
    "CWASHER": {"nome_coluna": "maquina_lavar", "info": "Máquina de lavar roupa"},
    "DRYER": {"nome_coluna": "secadora", "info": "Secadora de roupa"},
}

# df = pd.read_csv(csv_path) # Local
df = pd.read_csv(url_rec) # Colab

df = df[list(colunas)]
df = df.replace(-2, np.nan).dropna()  # Remover as linhas sem temperatura

df = df.rename(columns={k: v["nome_coluna"] for k, v in colunas.items()})

df["temp_inverno_noite_c"] = celsius(df["temp_inverno_noite_c"])
df["temp_inverno_dia_c"] = celsius(df["temp_inverno_dia_c"])

df["area_casa_m2"] = m_2(df["area_casa_m2"])

df = outliers_kwh(df)

print(df)

# pd.DataFrame(colunas.values()).to_csv("dados/colunas.csv", index=False)
# df.to_csv("dados/dados_pre_processado.csv", index=False)


            kwh  moradores  graus_dia_frio_f  graus_dia_calor_f  \
0      12521.48          2              3844               1679   
1       5243.05          1              3766               1458   
2       2387.64          1              3819               1696   
3       9275.07          2              2614               1718   
4       5869.70          2              4219               1363   
...         ...        ...               ...                ...   
18490  14333.77          4              3902               1383   
18491   5638.33          4              4572               1037   
18492   4425.20          1              9861                283   
18494  18604.35          1              1245               3038   
18495  19818.82          2              4423               1424   

       temp_inverno_dia_c  temp_inverno_noite_c  total_comodos  area_casa_m2  \
0               21.111111             20.000000              8       195.090   
1               21.111111          

In [4]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score, train_test_split

# ARQUIVO = "../dados/dados_pre_processado.csv"
ALVO = "kwh"
TEST_SIZE = 0.2
SEED = 42


def calcular_metricas(y_real, y_pred, n_features):
    n = len(y_real)
    r2 = r2_score(y_real, y_pred)
    r2_aj = 1 - (1 - r2) * (n - 1) / (n - n_features - 1)
    mse = mean_squared_error(y_real, y_pred)
    return {
        "R2": r2,
        "R2_ajustado": r2_aj,
        "MAE": mean_absolute_error(y_real, y_pred),
        "MSE": mse,
        "RMSE": np.sqrt(mse),
    }


def montar_equacao(modelo, colunas):
    termos = " ".join(f"{c:+.4f}*{n}" for n, c in zip(colunas, modelo.coef_))
    return f"{ALVO} = {modelo.intercept_:.4f} {termos}"


def treinar_modelo():
    # df = pd.read_csv(caminho)
    X = df.drop(columns=[ALVO])
    y = df[ALVO]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=SEED
    )

    modelo = LinearRegression()
    modelo.fit(X_train, y_train)

    y_pred_train = modelo.predict(X_train)
    y_pred_test = modelo.predict(X_test)
    k = X.shape[1]

    cv = cross_val_score(LinearRegression(), X, y, cv=5, scoring="r2")

    return {
        "modelo": modelo,
        "X_train": X_train, "X_test": X_test,
        "y_train": y_train, "y_test": y_test,
        "y_pred_train": y_pred_train, "y_pred_test": y_pred_test,
        "metricas": {
            "treino": calcular_metricas(y_train, y_pred_train, k),
            "teste": calcular_metricas(y_test, y_pred_test, k),
            "cv_r2_media": cv.mean(),
            "cv_r2_desvio": cv.std(),
        },
        "coeficientes": pd.Series(modelo.coef_, index=X.columns),
        "equacao": montar_equacao(modelo, X.columns),
    }


if __name__ == "__main__":
    r = treinar_modelo()
    m = r["metricas"]

    print("=" * 60)
    print("MÉTRICAS")
    print("=" * 60)
    print(f"{'':14}{'Treino':>14}{'Teste':>14}")
    for nome in ["R2", "R2_ajustado", "MAE", "MSE", "RMSE"]:
        print(f"{nome:14}{m['treino'][nome]:>14.4f}{m['teste'][nome]:>14.4f}")
    print(f"\nValidação cruzada (5 folds) R²: {m['cv_r2_media']:.4f} ± {m['cv_r2_desvio']:.4f}")

    print("\n" + "=" * 60)
    print("COEFICIENTES")
    print("=" * 60)
    print(f"Intercepto: {r['modelo'].intercept_:.4f}")
    print(r["coeficientes"].round(4).to_string())

    print("\n" + "=" * 60)
    print("EQUAÇÃO DA RETA (hiperplano)")
    print("=" * 60)
    print(r["equacao"])

MÉTRICAS
                      Treino         Teste
R2                    0.3512        0.3625
R2_ajustado           0.3505        0.3597
MAE                3486.6851     3421.5582
MSE            20149512.2171 19465777.9299
RMSE               4488.8208     4412.0038

Validação cruzada (5 folds) R²: 0.3523 ± 0.0053

COEFICIENTES
Intercepto: -787.9146
moradores                668.6088
graus_dia_frio_f          -0.1730
graus_dia_calor_f          0.7939
temp_inverno_dia_c       -59.3230
temp_inverno_noite_c     144.4775
total_comodos            106.7473
area_casa_m2              11.2987
ventiladores_teto        220.8719
tvs                      559.0681
geladeiras               628.3728
freezers                1350.0701
microondas              -103.3334
lava_louca                56.0808
maquina_lavar            183.0067
secadora                1716.1393

EQUAÇÃO DA RETA (hiperplano)
kwh = -787.9146 +668.6088*moradores -0.1730*graus_dia_frio_f +0.7939*graus_dia_calor_f -59.3230*temp_inverno